# Scientific figures and conditional compute estimate

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import csv,json,pickle,hashlib,sys,math
import numpy as np
from contract import ROOT as REPO,REFERENCE,CODE,PROV,read_json,write_json,sha,verify_inputs
PHASE=REFERENCE
ROOT=REPO/'outputs/analysis'
ROOT.mkdir(parents=True,exist_ok=True)
def csv_out(path,rows):
    with path.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def csvread(path):
    with Path(path).open() as f:return list(csv.DictReader(f))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
read=read_json
META=REPO/'evidence/baselines/solid_rally_development'

def generate_figures():
    FIG=ROOT/'figures'; FIG.mkdir(exist_ok=True)
    data=read(ROOT/'plot_data.json');episodes=csvread(ROOT/'evaluation_episodes.csv');colors={'TASK':'#2864a5','MAX':'#cc6d21','RANDOM':'#328267'}
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':130})
    def save(fig,name):
        fig.savefig(FIG/(name+'.png'),bbox_inches='tight');fig.savefig(FIG/(name+'.pdf'),bbox_inches='tight');plt.close(fig)
    fig,axs=plt.subplots(3,1,figsize=(10,7),sharex=True)
    for ax,c in zip(axs,colors):
        rows=data[c+'-3001'];t=[r['derived_game_seconds'] for r in rows]
        ax.step(t,[int(r['valid']) for r in rows],where='post',color='#777777',label='Valid q (held between pairs)')
        fresh=[r for r in rows if r['fresh']];ax.vlines([r['derived_game_seconds'] for r in fresh],0,[r['affect_impulse'] for r in fresh],color=colors[c],label='q impulse (potential MAX reward)')
        ax.scatter(t,[r['delivered_reward'] for r in rows],s=5,color='#222222',alpha=.55,label='Actually delivered reward')
        ax.axvspan(0,6,color='#dddddd',alpha=.4);ax.set_ylim(-.05,1.12);ax.set_ylabel(c);ax.grid(alpha=.15)
    axs[0].legend(loc='upper right',fontsize=8);axs[-1].set_xlabel('Derived in-game seconds (0.2 per decision)')
    fig.suptitle('Episode 3001: validity, pair impulses and delivered reward\nFirst valid pair at 6 s; no fabricated warm-up q');fig.tight_layout();save(fig,'01_timing_and_impulses')
    fig,axs=plt.subplots(3,1,figsize=(10,7),sharex=True)
    for ax,c in zip(axs,colors):
        rows=data[c+'-3001'];fresh=[r for r in rows if r['fresh']]
        ax.plot([r['derived_game_seconds'] for r in fresh],[r['q'] for r in fresh],'.-',color=colors[c],label='q at valid pair')
        ax.set_ylim(0,1);ax.set_ylabel(c+' q');ax.grid(alpha=.15)
        twin=ax.twinx();twin.plot([r['derived_game_seconds'] for r in rows],[r['raw_score_signal'] for r in rows],color='#333333',linestyle='--');twin.set_ylim(-.05,1.1);twin.set_ylabel('Raw task score')
    axs[-1].set_xlabel('Derived in-game seconds');fig.suptitle('Preference support and task score on the same clock\nEpisode 3001 selected in advance for each condition');fig.tight_layout();save(fig,'02_q_and_task_clock')
    fig,ax=plt.subplots(figsize=(9,4))
    for i,c in enumerate(colors):
        selected=[r for r in episodes if r['condition']==c]
        ax.scatter(i+np.linspace(-.2,.2,len(selected)),[float(r['raw_score']) for r in selected],s=40,color=colors[c])
        for x,r in zip(i+np.linspace(-.2,.2,len(selected)),selected):
            if float(r['raw_score'])>0:ax.annotate(r['requested_Unity_seed'],(x,float(r['raw_score'])),xytext=(0,8+14*(int(r['requested_Unity_seed'])%3)),textcoords='offset points',ha='center',fontsize=8)
    ax.set_xticks(range(3),list(colors));ax.set_ylabel('Final raw task score');ax.set_ylim(-.08,1.35);ax.grid(axis='y',alpha=.2)
    ax.set_title('All 30 development episode scores\nHorizontal offsets separate points; repetitions are not independent training replicates')
    fig.tight_layout();save(fig,'03_episode_task_scores')
    fig,axs=plt.subplots(2,2,figsize=(11,7));training_rows=[]
    for column,c in enumerate(['TASK','MAX']):
        rows=read(PHASE/f'runs/P1-{c}/attempt-01/train/episode_summaries.json');decisions=0;task=0;x=[];returns=[];cumulative=[]
        for r in rows:
            decisions+=r['decisions'];task+=r['task_return'];x.append(decisions);returns.append(r['delivered_return']);cumulative.append(task)
            training_rows.append({'condition':c,'episode':r['episode'],'actual_decisions_at_end':decisions,**{k:r[k] for k in ['decisions','task_return','delivered_return','raw_score','pairs','end_reason']}})
        axs[0,column].plot(x[:-1],returns[:-1],'.-',color=colors[c],linewidth=.8)
        axs[0,column].scatter(x[-1:],returns[-1:],marker='^',facecolors='none',edgecolors=colors[c],label='Final 200-decision fragment')
        axs[0,column].set_title(c+' delivered episode return');axs[0,column].set_ylabel('Task-event reward' if c=='TASK' else 'Sum of q impulses');axs[0,column].legend(fontsize=7)
        axs[1,column].step(x,cumulative,where='post',color=colors[c]);axs[1,column].set_ylabel('Cumulative task-progress events');axs[1,column].set_xlabel('Actual training decisions')
        for ax in axs[:,column]:ax.grid(alpha=.2);ax.set_xlim(0,52224)
    fig.suptitle('Pilot learning records at the fixed budget\nTop-row reward scales/objectives differ; bottom row uses a common task-progress event count');fig.tight_layout();save(fig,'04_training_progress')
    with (ROOT/'training_episode_series.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(training_rows[0]));w.writeheader();w.writerows(training_rows)

    # Rates include logged rollout collection, PPO updates and checkpoint writes.
    v={c:read(META/f'{c.lower()}_verification.json') for c in ['TASK','MAX']}
    rates={c:v[c]['learning_decisions_per_second'] for c in v};vslow=min(rates.values());vfast=max(rates.values())
    e={c:float(np.mean([float(r['worker_wall_seconds']) for r in episodes if r['condition']==c])) for c in colors}
    eloop={c:float(np.mean([float(r['observed_loop_seconds']) for r in episodes if r['condition']==c])) for c in colors}
    eoverhead={c:e[c]-eloop[c] for c in colors}
    startup={c:read(PHASE/f'runs/P1-{c}/attempt-01/train/result.json')['wall_seconds']-v[c]['learning_seconds'] for c in v}
    def estimate(T):
        training_known=5*T/rates['TASK']+5*T/rates['MAX']
        train=[training_known+15*T/vfast,training_known+15*T/vslow]
        overhead=[25*min(startup.values()),25*max(startup.values())]
        eval_known=150*e['TASK']+150*e['MAX']+30*e['RANDOM']
        evaluation=[eval_known+450*min(e['TASK'],e['MAX']),eval_known+450*max(e['TASK'],e['MAX'])]
        total=[train[i]+overhead[i]+evaluation[i] for i in [0,1]]
        return {'decisions_per_policy':T,'core_decisions':25*T,'training_loop_seconds_range':train,'training_nonloop_seconds_range':overhead,
            'evaluation_seconds_range':evaluation,'serial_core_seconds_range':total,'serial_core_hours_range':[x/3600 for x in total],
            'plus25percent_contingency_hours_range':[x*1.25/3600 for x in total]}
    storage={}
    for c in ['TASK','MAX']:
        d=PHASE/f'runs/P1-{c}/attempt-01/train'
        storage[c]={'training_events_bytes_per_decision':(d/'events.jsonl').stat().st_size/51200,
            'final_checkpoint_bytes':(d/'checkpoint_51200.zip').stat().st_size,
            'full_archive_compression_ratio':read(META/f'{c.lower()}_archive_verification.json')['archive_bytes']/read(META/f'{c.lower()}_archive_verification.json')['raw_bytes']}
    eval_bytes=[]
    for c in colors:
        for seed in range(3001,3011):
            d=PHASE/f'runs/P1-{c}/attempt-01/evaluate-{seed}';eval_bytes.append((d/'events.jsonl').stat().st_size)
    build_bytes=sum(p.stat().st_size for p in (CODE/'affectively/builds/solid/Linux').rglob('*') if p.is_file())
    raw_train=[25_000_000*min(s['training_events_bytes_per_decision'] for s in storage.values()),25_000_000*max(s['training_events_bytes_per_decision'] for s in storage.values())]
    raw_eval=[780*min(eval_bytes),780*max(eval_bytes)];checkpoint_bytes=25*math.ceil(1_000_000/10240)*max(s['final_checkpoint_bytes'] for s in storage.values())
    development={c:read(PHASE/f'runs/P1-{c}/attempt-01/result.json')['wall_seconds'] for c in colors}
    development['smoke_success']=read(PHASE/'runs/P1-TRAIN-SMOKE/attempt-02/result.json')['wall_seconds']
    development['smoke_failed']=read(PHASE/'runs/P1-TRAIN-SMOKE/attempt-01/result.json')['wall_seconds']
    result={'status':'CONDITIONAL_PLANNING_ESTIMATE','measured_training_rates':rates,'evaluation_worker_seconds_per_episode':e,
        'evaluation_loop_seconds_per_episode':eloop,'evaluation_nonloop_seconds_per_episode':eoverhead,'training_nonloop_seconds_per_policy':startup,
        'nominal':estimate(1_000_000),'rollout_aligned_sensitivity':estimate(1_001_472),'development_attempt_seconds':development,
        'storage':{'measured':storage,'raw_training_event_bytes_range':raw_train,'raw_evaluation_event_bytes_range':raw_eval,
            'checkpoint_bytes_at10240_interval':checkpoint_bytes,'one_build_bytes':build_bytes,'duplicated_build_bytes_for805_processes':805*build_bytes,
            'raw_event_plus_checkpoint_GiB_range':[(raw_train[i]+raw_eval[i]+checkpoint_bytes)/2**30 for i in [0,1]],
            'compression_limit':'Observed whole-pilot archives include varied file types; compression ratios are planning proxies, not guaranteed for target conditions.'},
        'assumptions':['25 fresh main policies: five each TASK/MAX/A/B/CONST; no controls dropped.','750 trained-policy evaluations plus30 RANDOM =780; current600-decision horizon, fresh process per episode.','A/B/CONST rates and evaluation cost bracketed by TASK/MAX, unmeasured until target integration.','Evaluation worker time already includes startup/reload/cleanup; do not add these twice.','Training nonloop residual combines imports/model initialization/player startup/final checks/cleanup; not isolated model-only timing.','25percent planning contingency separate from measured cost; development/failures listed separately.','Serial workstation estimate; no validated concurrency or acceleration; main seed/horizon design still requires G2.']}
    (ROOT/'compute_estimate.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result,indent=2))
print('Scientific figures and conditional compute estimate definitions/execution completed.')


Frozen runtime contract definitions/execution completed.


Scientific figures and conditional compute estimate definitions/execution completed.
